# Taylor diagrams

The `Taylor` diagnostic summarises in a single plot how well the spatial pattern of a climatology matches a reference ([Taylor, 2001](https://doi.org/10.1029/2000JD900719)).
Each point of the diagram shows, at once:

- the **correlation** with the reference (angle),
- the **standard deviation** of the pattern (radial distance), normalised by the one of the reference,
- the **centred root mean square difference** (CRMSD), i.e. the distance from the reference point (dashed green contours).

Normalising by the reference standard deviation makes the diagram dimensionless, so that many variables and models can be compared on the same plot.
The diagnostic follows the usual AQUA structure:

- `Taylor` retrieves one dataset, computes its annual and seasonal climatologies and, given a reference `Taylor` instance, the statistics. These are saved as NetCDF and can be loaded back with `load()`.
- `PlotTaylor` produces the diagrams from the statistics.
- `plot_taylor_diagram` and `compute_taylor_statistics` are the generic building blocks, usable on any data.

As a scientific case we evaluate the **phase 2 historical simulations** (`climatedt-gen2`, `baseline-hist`, LRA at 1° monthly) against ERA5 and CERES over 1990-2014.

*This notebook needs the phase 2 LRA, e.g. on LUMI.*

In [1]:
from aqua.diagnostics import PlotTaylor, Taylor
from aqua.diagnostics.taylor import compute_taylor_statistics, plot_taylor_diagram

/opt/homebrew/Caskroom/miniconda/base/envs/aqua-dev/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

The three coupled models of phase 2 and the reference. Data and reference must be on the same grid: the LRA is already at 1°, while ERA5 is regridded to `r100`.
The climatologies are computed over 1990-2014.

In [2]:
startdate = "1990-01-01"
enddate = "2014-12-31"

datasets = [
    {"catalog": "climatedt-gen2", "model": "IFS-NEMO-5km", "exp": "baseline-hist", "source": "lra-r100-monthly"},
    {"catalog": "climatedt-gen2", "model": "IFS-FESOM-5km", "exp": "baseline-hist", "source": "lra-r100-monthly"},
    {"catalog": "climatedt-gen2", "model": "ICON-5km", "exp": "baseline-hist", "source": "lra-r100-monthly"},
]
era5 = {"catalog": "obs", "model": "ECMWF", "exp": "era5", "source": "monthly", "regrid": "r100"}

outputdir = "./taylor_output"
loglevel = "WARNING"

## Multi-variable diagram against ERA5

For each variable the climatology of the reference is computed once and shared by all the models.
`run()` computes the annual climatology and, by default, the four seasonal ones. Formulae are supported, and `short_name` is used to name them.

In [4]:
variables = [
    {"var": "2t"},
    {"var": "msl"},
    {"var": "tcc"},
    {"var": "tprate", "units": "mm/day"},
    {"var": "(10u^2+10v^2)^0.5", "formula": True, "short_name": "10si", "long_name": "Wind speed at 10m"},
]

statistics = []
for kwargs in variables:
    reference = Taylor(**era5, startdate=startdate, enddate=enddate, loglevel=loglevel)
    reference.run(**kwargs)

    for dataset in datasets:
        taylor = Taylor(**dataset, startdate=startdate, enddate=enddate, loglevel=loglevel)
        taylor.run(**kwargs)
        taylor.compute_statistics(reference=reference, outputdir=outputdir)
        statistics.append(taylor.statistics)

NoDataError: Some NetCDF files are missing for ECMWF era5 monthly, please check the url: ['/pfs/lustrep4/projappl/project_465002727/aqua/aqua-dvc/datasets/ERA5/mon/ERA5_*_mon_full*.nc']

The statistics are a small `xarray.Dataset` per model and variable, with one value per period. Besides the normalised quantities plotted in the diagram, they include the mean bias and the full RMSE, with `rmse² = bias² + crmse²`.

In [ ]:
statistics[0].to_dataframe()

By default the colors distinguish the variables and the markers the models. Points closer to the reference star are better: they have the right spatial variability and are well correlated with ERA5.

In [ ]:
plot = PlotTaylor(outputdir=outputdir, save_format=[], loglevel=loglevel)
fig, ax = plot.plot_taylor(statistics, period="annual", color_by="var", marker_by="dataset")

### Seasonal diagrams

The grouping keys can be changed: here, for a single variable, the colors are the seasons. This shows which seasons each model struggles with.

In [ ]:
stats_2t = [s for s in statistics if s.attrs["short_name"] == "2t"]
fig, ax = plot.plot_taylor(stats_2t, period="seasonal", color_by="period", marker_by="dataset")

With a single variable and period the diagram can also be in physical units, with `normalize=False`.

In [ ]:
fig, ax = plot.plot_taylor(stats_2t, period="JJA", color_by="dataset", marker_by="var", normalize=False)

## Radiation at the top of the atmosphere against CERES

The same machinery works with any reference. Here CERES EBAF is used for the outgoing radiation and the cloud radiative effects, over the overlapping period 2001-2014.

In [ ]:
ceres = {"catalog": "obs", "model": "CERES", "exp": "ebaf-toa421", "source": "monthly", "regrid": "r100"}
radiation = [
    {"var": "tnlwrf"},
    {"var": "tnswrf"},
    {"var": "tnlwrf-tnlwrfcs", "formula": True, "short_name": "cre_toa_lw"},
    {"var": "tnswrf-tnswrfcs", "formula": True, "short_name": "cre_toa_sw"},
]

statistics_rad = []
for kwargs in radiation:
    reference = Taylor(**ceres, startdate="2001-01-01", enddate=enddate, loglevel=loglevel)
    reference.run(**kwargs, seasonal=False)
    for dataset in datasets:
        taylor = Taylor(**dataset, startdate="2001-01-01", enddate=enddate, loglevel=loglevel)
        taylor.run(**kwargs, seasonal=False)
        taylor.compute_statistics(reference=reference, outputdir=outputdir)
        statistics_rad.append(taylor.statistics)

fig, ax = plot.plot_taylor(statistics_rad, annotate=True)

## Regions

A region from the centralised regions file restricts the domain, e.g. the Tropics.

In [ ]:
reference = Taylor(**era5, startdate=startdate, enddate=enddate, region="tropics", loglevel=loglevel)
reference.run(var="tprate", units="mm/day")

stats_tropics = []
for dataset in datasets:
    taylor = Taylor(**dataset, startdate=startdate, enddate=enddate, region="tropics", loglevel=loglevel)
    taylor.run(var="tprate", units="mm/day")
    taylor.compute_statistics(reference=reference, outputdir=outputdir)
    stats_tropics.append(taylor.statistics)

fig, ax = plot.plot_taylor(stats_tropics, period="seasonal", color_by="period", marker_by="dataset")

## Plotting from previous results

The statistics saved as NetCDF can be loaded without retrieving any data, e.g. to change the plot.
The reference instance only provides the identity of the reference: it does not need to be run.

In [ ]:
reference = Taylor(**era5, loglevel=loglevel)
loaded = []
for dataset in datasets:
    taylor = Taylor(**dataset, startdate=startdate, enddate=enddate, loglevel=loglevel)
    taylor.load(var="2t", reference=reference, outputdir=outputdir)
    loaded.append(taylor.statistics)

fig, ax = plot.plot_taylor(loaded, std_max=1.5)

## Generic building blocks

`compute_taylor_statistics` works on any pair of fields on the same grid, over any set of dimensions.
As an example, we evaluate the **annual cycle** as a space-time pattern, i.e. computing the statistics over the 12 monthly climatologies and space together, a common metric in model evaluation (e.g. the PCMDI metrics package).
Cell areas are used as weights.

In [ ]:
reference = Taylor(**era5, startdate=startdate, enddate=enddate, loglevel=loglevel)
reference.retrieve(var="2t")
cycle_ref = reference.data.groupby("time.month").mean()

std, corr, labels = [], [], []
for dataset in datasets:
    taylor = Taylor(**dataset, startdate=startdate, enddate=enddate, loglevel=loglevel)
    taylor.retrieve(var="2t")
    cycle = taylor.data.groupby("time.month").mean()
    stats = compute_taylor_statistics(cycle, cycle_ref, weights=taylor.area, dims=["month", "lat", "lon"])
    std.append(float(stats["std_norm"]))
    corr.append(float(stats["corr"]))
    labels.append(dataset["model"])

`plot_taylor_diagram` is the graphics function, which only needs standard deviations and correlations. It is designed to be moved to `aqua.core.graphics`.

In [ ]:
fig, ax = plot_taylor_diagram(
    std=std,
    corr=corr,
    labels=labels,
    markers=["o", "s", "^"],
    ref_label="ERA5",
    std_label="Normalised standard deviation",
    title="Annual cycle of 2m temperature, 1990-2014",
)